# Notebook 13 — Causal pruning validation (even/odd MLP)

One pruning validation result for the paper: on the even/odd MLP (`SimpleMLP` 784→8→4→2, MNIST digits {0,1,3,4}), does zeroing the weights BFT ranks **most important** for a class circuit hurt *that class* more than the bystander class, and more than the standard baselines?

Per (seed, target class) observation it runs `ablation_sweep` over a fraction grid with six rankings — `bft_top`, `bft_bottom`, `magnitude`, `act_magnitude`, `taylor`, `random` — evaluated as per-class accuracy on the full filtered MNIST test set. 5 model seeds × 2 target classes = 10 paired observations for the error bars and tests.

Follows the notebook-09/11 cluster conventions: one execution = the whole experiment, `NB13_MODE` (`local` | `cluster`) picks the compute profile, and every seed **checkpoints** to `data/results/nb13_pruning_mlp_even_odd.json` so an interrupted run is still usable. The JSON carries the aggregated curves (mean ± sd, including the fraction-0 baseline point) plus every raw per-observation curve, so the paper figure can be drawn from it alone. See the final cell for how to run on the cluster.

## §0 · Setup, compute profile & helpers

In [ ]:
import os, sys, json, warnings
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.transforms import ToTensor
from scipy.stats import wilcoxon, ttest_rel

from src import (load_experiment, get_transform, get_loaders_from_config,
                 collect_layer_dicts, bft, ablation_sweep)
from src.training import label_transform_even_odd
from src.data_utils import label_transformed_loader

warnings.filterwarnings('ignore')
REPO = os.path.abspath('..')                     # notebook lives in notebooks/
MODE = 'cluster'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      ('mps' if torch.backends.mps.is_available() else 'cpu'))

FIG_DIR    = os.path.join(REPO, 'figs', '13_pruning_mlp_even_odd')
RES_DIR    = os.path.join(REPO, 'data', 'results')
MODEL_ROOT = os.path.join(REPO, 'data', 'models')
for d in (FIG_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)

# Compute profile. 'local' is a laptop smoke test — numbers are NOT publication-grade.
if MODE == 'cluster':
    N_TRACE          = None      # trace on all correctly-classified test samples
    BFT_MAX_ITER     = None      # -> bft default (500), matching notebooks 01-05
    N_RANDOM_REPEATS = 10
    SEED_SUBSET      = range(5)
    FRACTIONS        = [0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]
else:
    N_TRACE          = 600
    BFT_MAX_ITER     = 120
    N_RANDOM_REPEATS = 3
    SEED_SUBSET      = range(2)
    FRACTIONS        = [0.05, 0.20, 0.40]

FRAC_STAT = 0.20                 # fraction the significance tests are run at

results = {'experiment': 'mlp_even_odd_pruning', 'mode': MODE}
COMPLETED = []
RESULT_PATH = os.path.join(RES_DIR, 'nb13_pruning_mlp_even_odd.json')


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.floating, np.integer)):
        return float(o)
    return o if isinstance(o, (float, int, str, bool)) or o is None else str(o)


def checkpoint(section=None):
    """Atomically write results to disk so a partial run is not wasted."""
    if section and section not in COMPLETED:
        COMPLETED.append(section)
    results['completed_sections'] = list(COMPLETED)
    tmp = RESULT_PATH + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(jsonable(results), f, indent=1)
    os.replace(tmp, RESULT_PATH)
    print(f'  [checkpoint] {section or ""} -> {os.path.relpath(RESULT_PATH, REPO)} '
          f'({len(COMPLETED)} sections)')


def _bft_iter():
    return {} if BFT_MAX_ITER is None else {'max_iter': BFT_MAX_ITER}


print(f'MODE={MODE}  DEVICE={DEVICE}  N_TRACE={N_TRACE}  '
      f'fractions={FRACTIONS}  seeds={list(SEED_SUBSET)}')

## §1 · Config — PUBLICATION SETTINGS

BFT hyperparameters copied verbatim from notebook 01 §1 (the nb09 S10 sweep winner, `rank ×0.7`; `[4, 2, 2]` is the floored profile that actually executes — see PUBLICATION_SETTINGS.md). If notebook 01's config changes, update here.

In [ ]:
EXP_BASE     = 'mnist_even_odd_mlp_8_4_0134'
DIGIT_FILTER = [0, 1, 3, 4]
N_CLASSES    = 2
CLASS_NAMES  = {0: 'even', 1: 'odd'}

K_MAX_PER_LAYER    = [4, 2, 2]
N_BRANCHES         = [1, 1, 2]
STIMULUS_THRESHOLD = 0.5

ABL_METHODS = ['bft_top', 'bft_bottom', 'magnitude', 'act_magnitude', 'taylor', 'random']

results['config'] = {
    'ckpt': EXP_BASE, 'digit_filter': DIGIT_FILTER, 'class_names': CLASS_NAMES,
    'k_max': K_MAX_PER_LAYER, 'n_branches': N_BRANCHES,
    'stimulus_threshold': STIMULUS_THRESHOLD,
    'fractions': FRACTIONS, 'frac_stat': FRAC_STAT, 'methods': ABL_METHODS,
    'n_random_repeats': N_RANDOM_REPEATS, 'seeds': list(SEED_SUBSET),
    'n_trace': N_TRACE, 'bft_max_iter': BFT_MAX_ITER, 'device': str(DEVICE)}

# Ablation evaluation set: the FULL filtered MNIST test set (independent of any trace cap),
# scored per class via label_transform_even_odd — same as notebook 08 §7.
mnist_test = datasets.MNIST('../data/', train=False, download=True, transform=ToTensor())
mask = np.isin(np.array(mnist_test.targets), DIGIT_FILTER)
eval_loader = DataLoader(Subset(mnist_test, np.where(mask)[0].tolist()),
                         batch_size=256, shuffle=False)
print(f'eval set: {int(mask.sum())} samples (digits {DIGIT_FILTER})')

## §2 · Per-seed BFT trace + ablation sweep

For each seed: rebuild the notebook-01 trace (same loaders, same publication HPs), then run `ablation_sweep` for both target classes. `act_magnitude` needs the per-layer inputs, which come from the same `collect_layer_dicts` call that defines the trace sample set. Checkpoints after every seed.

In [ ]:
per_obs = []
results['per_obs'] = per_obs

for seed in SEED_SUBSET:
    ed = os.path.join(MODEL_ROOT, f'{EXP_BASE}_seed{seed}')
    if not os.path.exists(os.path.join(ed, 'weights.pt')):
        print(f'seed {seed}: no checkpoint under {ed} — skipped')
        continue
    model, config = load_experiment(ed, DEVICE)
    _, test_loader = get_loaders_from_config(config)
    label_transform = get_transform(config['label_transform'])
    tl = test_loader if N_TRACE is None else DataLoader(
        Subset(test_loader.dataset, list(range(min(N_TRACE, len(test_loader.dataset))))),
        batch_size=256, shuffle=False)

    coll = collect_layer_dicts(model, tl, label_transform=label_transform, device=DEVICE)
    layer_inputs = [d['input_fmap'] for d in coll['layer_data']]
    vloader = label_transformed_loader(tl, label_transform)
    tree = bft(model, vloader, k_max=K_MAX_PER_LAYER, n_branches=N_BRANCHES,
               stimulus_threshold=STIMULUS_THRESHOLD, weighting='img_selectivity',
               n_jobs=3, **_bft_iter())

    for d in range(N_CLASSES):
        ab = ablation_sweep(model, tree, eval_loader, target_class=d,
                            fractions=FRACTIONS, methods=ABL_METHODS,
                            label_transform=label_transform_even_odd, device=DEVICE,
                            n_random_repeats=N_RANDOM_REPEATS,
                            layer_inputs_list=layer_inputs, verbose=0)
        per_obs.append({'seed': seed, 'target_class': d,
                        'baseline': ab.baseline,
                        'bft_info': {k: ab.bft_info[k] for k in
                                     ('k_star', 'selectivity', 'is_selective', 'warning')},
                        'curves': ab.results})
        tdrop = ab.baseline[d] - ab.results['bft_top'][FRAC_STAT][d]
        print(f'  seed {seed}  class {CLASS_NAMES[d]}: baseline={ab.baseline[d]:.3f}  '
              f'bft_top drop@{FRAC_STAT:.2f}={tdrop:.3f}'
              + ('' if ab.bft_info['is_selective'] else '  [WARN: no selective factor]'))
    checkpoint(f'seed{seed}')

print(f'{len(per_obs)} observations')

## §3 · Aggregate curves + significance

Per method: mean ± sd across the (seed, class) observations of **target-class accuracy** and **bystander accuracy** at each fraction, prepended with the fraction-0 baseline point — everything the paper figure needs.

Tests (paired over the 10 observations, at `FRAC_STAT` / target-drop AUC over the grid): `bft_top` target vs bystander drop (specificity, Wilcoxon), and `bft_top` vs each baseline ranking on target AUC (paired t-test).

In [ ]:
def obs_curves(o, method):
    """(target_accs, bystander_accs) over [0]+FRACTIONS for one observation."""
    d = o['target_class']
    others = [c for c in range(N_CLASSES) if c != d]
    t = [o['baseline'][d]] + [o['curves'][method][f][d] for f in FRACTIONS]
    b = [np.mean([o['baseline'][c] for c in others])] + \
        [np.mean([o['curves'][method][f][c] for c in others]) for f in FRACTIONS]
    return np.array(t), np.array(b)


agg = {'fractions': [0.0] + list(FRACTIONS), 'n_obs': len(per_obs), 'methods': {}}
for m in ABL_METHODS:
    T = np.stack([obs_curves(o, m)[0] for o in per_obs])
    B = np.stack([obs_curves(o, m)[1] for o in per_obs])
    agg['methods'][m] = {
        'target_mean': T.mean(0), 'target_sd': T.std(0),
        'bystander_mean': B.mean(0), 'bystander_sd': B.std(0)}
results['aggregate'] = agg

# per-observation summary numbers the tests run on
drops = {m: {'target': [], 'bystander': [], 'auc': []} for m in ABL_METHODS}
for o in per_obs:
    d = o['target_class']
    base_t = o['baseline'][d]
    others = [c for c in range(N_CLASSES) if c != d]
    for m in ABL_METHODS:
        drops[m]['target'].append(base_t - o['curves'][m][FRAC_STAT][d])
        drops[m]['bystander'].append(np.mean(
            [o['baseline'][c] - o['curves'][m][FRAC_STAT][c] for c in others]))
        drops[m]['auc'].append(np.mean(
            [base_t - o['curves'][m][f][d] for f in FRACTIONS]))

stats = {'frac_stat': FRAC_STAT, 'n_obs': len(per_obs),
         'drops': drops, 'tests': {}}
t_bt, b_bt = np.array(drops['bft_top']['target']), np.array(drops['bft_top']['bystander'])
if len(t_bt) >= 2 and np.any(t_bt != b_bt):
    w, p = wilcoxon(t_bt, b_bt)
    stats['tests']['bft_top_target_vs_bystander'] = {'wilcoxon_stat': float(w), 'p': float(p)}
for m in ABL_METHODS:
    if m == 'bft_top' or len(per_obs) < 2:
        continue
    t, p = ttest_rel(drops['bft_top']['auc'], drops[m]['auc'])
    stats['tests'][f'bft_top_vs_{m}_target_auc'] = {'t': float(t), 'p': float(p)}
results['stats'] = stats
checkpoint('aggregate')

print(f"n_obs={len(per_obs)}  (target drop @{FRAC_STAT}, mean over obs)")
for m in ABL_METHODS:
    print(f"  {m:14s} target={np.mean(drops[m]['target']):+.3f}  "
          f"bystander={np.mean(drops[m]['bystander']):+.3f}")
for k, v in stats['tests'].items():
    print(f"  {k}: {v}")

## §4 · Preview figure

Quick look at the result (the paper version gets restyled from the JSON): target-class accuracy vs pruned fraction per ranking (left) and bystander accuracy (right). Specific circuit pruning should show `bft_top` dropping the target class fastest while leaving the bystander class comparatively intact.

In [ ]:
COLORS = {'bft_top': '#e15759', 'bft_bottom': '#f28e2b', 'magnitude': '#76b7b2',
          'act_magnitude': '#59a14f', 'taylor': '#af7aa1', 'random': '#333333'}
LABELS = {'bft_top': 'BFT most important', 'bft_bottom': 'BFT least important',
          'magnitude': 'Magnitude', 'act_magnitude': 'Act. magnitude',
          'taylor': 'Taylor', 'random': 'Random'}

fx = np.array(agg['fractions'])
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
for ax, key, title in [(axes[0], 'target', 'target class'),
                       (axes[1], 'bystander', 'bystander class')]:
    for m in ABL_METHODS:
        mu = np.asarray(agg['methods'][m][f'{key}_mean'])
        sd = np.asarray(agg['methods'][m][f'{key}_sd'])
        ax.plot(fx, mu, '-o', ms=3, lw=1.4, color=COLORS[m], label=LABELS[m])
        ax.fill_between(fx, mu - sd, mu + sd, color=COLORS[m], alpha=0.15, lw=0)
    ax.set(xlabel='fraction of weights pruned', title=title, ylim=(0, 1.05))
axes[0].set_ylabel('accuracy')
axes[1].legend(fontsize=7, loc='lower left')
fig.suptitle(f'Class-circuit pruning — even/odd MLP ({agg["n_obs"]} seed×class obs, '
             f'mean ± sd)', y=1.03)
fig.tight_layout()
p = os.path.join(FIG_DIR, 'fig_pruning_preview.png')
fig.savefig(p, bbox_inches='tight')
print('saved', os.path.relpath(p, REPO))
plt.show()

## How to run on the cluster

`NB13_MODE=cluster` turns off the laptop caps (all trace samples, bft's default 500 NMF iters, all 5 seeds, the full 8-point fraction grid, 10 random repeats). CPU is fine — the model is tiny; expect minutes, not hours.

```bash
cd notebooks
NB13_MODE=cluster \
  ../.venv/bin/python -m nbconvert --to notebook --execute \
  --output executed_13_pruning_mlp_even_odd.ipynb \
  --ExecutePreprocessor.timeout=100000 13_pruning_mlp_even_odd.ipynb
```

Needs the `mnist_even_odd_mlp_8_4_0134_seed{0..4}` checkpoints under `data/models/` (train via `scripts/train_extra_seeds.sh` if missing); MNIST downloads automatically.

Output: `data/results/nb13_pruning_mlp_even_odd.json`. Copy it into `logs/results/` alongside the nb09 output and plot from `results['aggregate']` (fractions include the 0-point baseline; JSON round-trip turns the raw `per_obs` curve keys into strings — the aggregate arrays are the ones to draw from).